# Volatility Forecasting: From FNN to LSTM

## Objective

The goal of this project is to show the limitation of standard model in forecasting the volatility of the S&P 500 using three neural network architectures of increasing complexity: a **Feedforward Neural Network (FNN)**, a **Recurrent Neural Network (RNN)**, and a **Long Short-Term Memory network (LSTM)**.

Rather than simply comparing final performance, the goal is to understand *why* each architecture improves on the previous one by demonstrating their limitations empirically. Each model is studied in depth, and its weaknesses are shown through concrete experiments before motivating the move to the next architecture.

---

## What is Volatility?

Volatility measures the degree of variation in an asset's returns over time. It is one of the most important quantities in finance — it drives option pricing, risk management, and portfolio allocation decisions.

Volatility is not directly observable, so we approximate it using **realized volatility**: the rolling standard deviation of daily returns over a given window.

One of the most well-known properties of financial volatility is **volatility clustering** — periods of high volatility tend to be followed by more high volatility, and calm periods tend to persist. Any model that claims to forecast volatility must be able to capture this dynamic.

---

## Dataset

We use daily returns of the **S&P 500 (SPY ETF)** from 1993 to 2026. From these returns we construct:

- **Features (X):** the N previous daily returns, used as a lagged window of inputs
- **Target (y):** the realized volatility of the next period, computed as the rolling standard deviation of returns

The dataset is split into a training set and a test set, preserving temporal order — no shuffling across the split, as this would constitute a look-ahead bias.

---

## Architecture 1: Feedforward Neural Network (FNN)

### What it does

A FNN takes a fixed-size window of past returns as input and maps it directly to a volatility prediction. It has no internal memory — each prediction is made independently from a flat vector of features.

### What we study

- Baseline performance on the forecasting task
- Sensitivity to window size: does performance improve as we feed more past data?
- Behavior under temporal shuffling: does the model care about the order of its inputs?
- Performance breakdown during volatility regime changes (calm vs crisis periods)

### Limitations we demonstrate

**Order blindness:** shuffling the rows of the training set produces nearly identical results, proving the model has no understanding of temporal order.

**Fixed context:** performance plateaus or degrades beyond a certain window size because the network treats all lags equally — it has no mechanism to weight recent information more than distant information.

**Regime blindness:** the model fails during sudden volatility spikes because it sees only a static snapshot of the past, with no persistent state capturing the dynamic buildup leading into a crisis.



## Evaluation

Models are evaluated using the following metrics:

- **MSE** (Mean Squared Error): standard regression loss
- **QLIKE**: a loss function standard in the volatility forecasting literature, more sensitive to underprediction of large spikes
- **Performance by regime**: metrics computed separately on calm and crisis periods to expose where each model breaks down



The progression from FNN to RNN to LSTM is not just a performance comparison — it is a story about what information each architecture is capable of using, and what structural properties of financial time series each one fails to exploit.

### 1 - Download the data
 - The dataset is constructed this way: 
    - X: LAGS past returns
    - Y: following day volatility
    - Try to do a small analysis, can we actually understand things from it or no (the goal is no otherwise no need for ML)
    - Try to identify regimes in vol (do graph representation), spikes, drawdown etc

<b> !!! For graphs always use dark background (c'est plus stylé) </b>

In [1]:
import os
import sys
sys.path.append('../src')
from data_utils import *
from sklearn.model_selection import train_test_split

DATA_PATH = r"../data/df_sp_500_log_ret.csv"
# df = download_sp500_log_return_series()
# df.to_csv(DATA_PATH)
df = pd.read_csv(DATA_PATH, index_col='Date')

"""Size of the window, use lags past returns to predict following volatility """

LAGS = 10
#X, y = build_dataset_returns(df)
X, y = build_dataset_abs_returns(df, LAGS)
X, y, scaler_x, scaler_y = scale_data(X,y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=80718)


### 2- Standard Linear Regression:
 <b> Steps to do</b>
- Train a standard linear regression on the dataset
- Get the metrics, performance, significance
- Analysis of the feature the model is actually looking at, then try with new feature      (squared feature, ratio between some days etc)
- Is the model always predicting zero? 
- Correlation between the features?
- Limitation of the model?
- option: try a simple arch garch model

*all functions used to do so should be computed in the /model_analysis/regression.py file*


In [2]:
from model_analysis import *
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

lin_reg = LinearRegression().fit(X_train, y_train)
preds = lin_reg.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, preds):.4f}")

MAE: 0.6377


### 3- FNN (chapter 4 of the pdf is really helpful)

- Build a simple FNN
- Try different architecture, depth, weight initialization, loss function(to be implemented in dl_utils)
- Look at what he is doing better than the regression, try to fine tune it, maybe plot weights vs loss (for models with few weights see p.101 of the paper) for example surface of x,y are 2 weights, z is the loss. Or plot loss for different parameters (for example momentum of SGD momentum), don t forget  to fine tune the lr decay, very important
- Try to do histogram of the wieghts (each layer) for different result/configuration, one that works well and when that doesn t for example
- Try to plot variation of histograms of weight during training
- Try to do visualisation for the gradient, does he get stuck etc??
- Try to analyse the the feature the model is computing (read the relevent dl_from_scratch part), for a simple architecture should be the result of the matrix operations, discuss activation function, dropout, optimizer and all model parametersnn analyse trade off between fast and slow learning (learning rate), can the model get stuck etc ...
- Try to interpret the model using, SHAP or LIME (lime preferable)
- <u> Important part</u> WShow the limitation of the model, what the model can t modelise (sequence order, try shuffling the feature of the datas for example), is he good with regimles (shouldn t be) 
- Conclude on :
    - what is the best configuration activation, initialization, learning rate decay etc, try to interpret
    - what we need, seqence knwoledge, memoriezation etc hence rnn -> LSTM

*always check that the custom implem (with dl utils) closely matches the result from a pytorch implementation*

*u may have to modify the function inside the dl_utils module to monitor the difference evolution, contact <u><b>Lord Edouard</b></u> if too complicated*

*essaie de lire des trucs sur les FNN avant quand même*

In [4]:
""" Custom Implementation"""
from dl_utils import *
neural_network = NeuralNetwork(                                                                                                                                                                           
    layers=[                                                                                                                                                                                              
        Dense(neurons=32, activation=ReLU(), dropout = 0.5, weight_init='he_normal'),                                                                                                                                           
        Dense(neurons=16, activation=ReLU(), weight_init='he_normal'),                                                                                                                                           
        Dense(neurons=1,  activation=Linear())  
    ],                                                                                                                                                                                                    
    loss=MeanSquaredError()                                                                                                                                                                             
) 
trainer = Trainer(neural_network, SGD(lr=0.001))

trainer.fit(X_train, y_train, X_test, y_test,                                                                                                                                                             
    epochs=300,                                                                                                                                                                                           
    eval_every=10,  
    seed = 201906501,                                                                                                                                                                                     
    patience=5)
nn_preds = neural_network.forward(X_test)                                                                                                                                                                 
print(f"NN  MAE: {np.mean(np.abs(nn_preds - y_test)):.4f}")    

Validation loss after 10 epochs is 0.850816
No improvement after epoch 20 (1/5), best loss: 0.850816
Validation loss after 30 epochs is 0.810897
Validation loss after 40 epochs is 0.799861
Validation loss after 50 epochs is 0.792472
Validation loss after 60 epochs is 0.783144
No improvement after epoch 70 (1/5), best loss: 0.783144
No improvement after epoch 80 (2/5), best loss: 0.783144
Validation loss after 90 epochs is 0.779511
No improvement after epoch 100 (1/5), best loss: 0.779511
No improvement after epoch 110 (2/5), best loss: 0.779511
Validation loss after 120 epochs is 0.775429
Validation loss after 130 epochs is 0.771314
Validation loss after 140 epochs is 0.764617
No improvement after epoch 150 (1/5), best loss: 0.764617
No improvement after epoch 160 (2/5), best loss: 0.764617
No improvement after epoch 170 (3/5), best loss: 0.764617
No improvement after epoch 180 (4/5), best loss: 0.764617
Validation loss after 190 epochs is 0.763318
Validation loss after 200 epochs is 0

In [ ]:
""" Pytorch implementation"""

import torch
import torch.nn as nn


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

model = Net()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
loader  = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(100):
    model.train()
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_preds = model(X_test_t).numpy()

mae_torch = np.mean(np.abs(test_preds - y_test))
print(f"PyTorch MAE: {mae_torch:.4f}")

PyTorch MAE: 0.5792
